In [ ]:
import transformers

!pip install "torch" tensorboard -q
!pip install --upgrade "sentence-transformers" "datasets" "huggingface_hub" "transformers" -q
!pip install "pytrec_eval" -q

print(f"Transformers version after installation: {transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Transformers version after installation: 4.57.3


In [ ]:
import os
import torch
import warnings
from datasets import load_dataset, concatenate_datasets
from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.util import cos_sim
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

All libraries imported successfully!
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


model configuration and Matryoshka dimensions

In [ ]:
MODEL_ID = "BAAI/bge-base-en-v1.5"  
OUTPUT_DIR = "bge-base-financial-matryoshka"  

MATRYOSHKA_DIMENSIONS = [768, 512, 256, 128, 64]

NUM_EPOCHS = 4
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-5

print(f"Base Model: {MODEL_ID}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Matryoshka Dimensions: {MATRYOSHKA_DIMENSIONS}")
print(f"Training Epochs: {NUM_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

Base Model: BAAI/bge-base-en-v1.5
Output Directory: bge-base-financial-matryoshka
Matryoshka Dimensions: [768, 512, 256, 128, 64]
Training Epochs: 4
Batch Size: 32
Effective Batch Size: 512


Financial Q&A dataset with question-context pairs from NVIDIA's SEC filings.

### 3.1: Load the Dataset

**TODO**: Load the financial embedding dataset and prepare it for training.

**Dataset format**: Each sample has a `question` and `context` field.

In [ ]:
print("Loading dataset...")

dataset = load_dataset("zeitgeist-ai/financial-rag-nvidia-sec", split="train")

print(f"Dataset loaded with {len(dataset)} samples")
print(f"\nDataset structure: {dataset}")
print(f"\nDataset features: {dataset.features}")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dataset loaded with 7000 samples

Dataset structure: Dataset({
    features: ['question', 'answer', 'context', 'ticker', 'filing'],
    num_rows: 7000
})

Dataset features: {'question': Value('string'), 'answer': Value('string'), 'context': Value('string'), 'ticker': Value('string'), 'filing': Value('string')}


In [ ]:
print("Sample question-context pairs:\n")
for i in range(3):
    sample = dataset[i]
    print(f"Example {i+1}:")
    print(f"Question: {sample['question']}")
    print(f"Context: {sample['context'][:200]}...")  # Show first 200 chars
    print("-" * 80)
    print()

Sample question-context pairs:

Example 1:
Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?
Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields....
--------------------------------------------------------------------------------

Example 2:
Question: What are some of the recent applications of GPU-powered deep learning as mentioned by NVIDIA?
Context: Some of the most recent applications of GPU-powered deep learning include recommendation systems, which are AI algorithms trained to understand the preferences, previous decisions, and characteristics...
--------------------------------------------------------------------------------

Example 3:
Question: What significant invention did NVIDIA create in 1999?
Context: Our invention of the GPU in 1999 defined modern computer graphics and established NVIDIA as the leader in computer graphics....
--

In [ ]:
dataset = dataset.rename_column("question", "anchor")
dataset = dataset.rename_column("context", "positive")
dataset = dataset.add_column("id", [str(i) for i in range(len(dataset))])

print("Dataset prepared!")
print(f"\nUpdated columns: {dataset.column_names}")

Dataset prepared!

Updated columns: ['anchor', 'answer', 'positive', 'ticker', 'filing', 'id']


train/test

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

print("Dataset split complete!")
print(f"\nDataset splits:")
print(f"Training: {len(dataset['train'])} samples")
print(f"Test: {len(dataset['test'])} samples")

dataset["train"].to_json("train_dataset.json", orient="records")
dataset["test"].to_json("test_dataset.json", orient="records")
print("\nDatasets saved to disk!")

Dataset split complete!

Dataset splits:
Training: 6300 samples
Test: 700 samples


Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


Datasets saved to disk!


Load the base embedding model for evaluation.

In [ ]:
print("Loading pre-trained model...")
model = SentenceTransformer(MODEL_ID)
print(f"Model loaded: {MODEL_ID}")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading pre-trained model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: BAAI/bge-base-en-v1.5
   Embedding dimension: 768


In [ ]:
test_dataset = dataset['test'] 

train_dataset = dataset['train'] 

corpus_dataset = concatenate_datasets([train_dataset, test_dataset])
corpus = {str(sample['id']): sample['positive'] for sample in corpus_dataset}
queries = {str(sample['id']): sample['anchor'] for sample in test_dataset}

relevant_docs = {}
for q_id in queries:
    relevant_docs[q_id] = [q_id]  
    
print("Evaluation data prepared!")
print(f"   Corpus size: {len(corpus)} documents")
print(f"   Queries: {len(queries)} questions")

Evaluation data prepared!
   Corpus size: 7000 documents
   Queries: 700 questions


Create evaluators for each Matryoshka dimension.

In [ ]:
import pytrec_eval
from sentence_transformers.util import cos_sim

Evaluate the pre-trained model to establish our baseline.

In [ ]:
print("Evaluating baseline model...\n")
print("This may take a few minutes...\n")

baseline_scores = {}

full_query_embeddings = model.encode(list(queries.values()), convert_to_tensor=True)
full_corpus_embeddings = model.encode(list(corpus.values()), convert_to_tensor=True)

corpus_id_to_idx = {cid: idx for idx, cid in enumerate(corpus.keys())}

qrels = {}
for qid, rel_docs_list in relevant_docs.items():
    qrels[qid] = {doc_id: 1 for doc_id in rel_docs_list}

for dim in MATRYOSHKA_DIMENSIONS:
    print(f"  Evaluating for dimension {dim}...")

    query_embeddings_truncated = full_query_embeddings[:, :dim]
    corpus_embeddings_truncated = full_corpus_embeddings[:, :dim]

    sim_scores = cos_sim(query_embeddings_truncated, corpus_embeddings_truncated)

    run = {}
    for q_idx, qid in enumerate(queries.keys()):
        run[qid] = {
            corpus_id: sim_scores[q_idx, corpus_id_to_idx[corpus_id]].item()
            for corpus_id in corpus.keys()
        }

    evaluator_metrics = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'})
    scores = evaluator_metrics.evaluate(run)

    mean_ndcg_at_10 = sum([s['ndcg_cut_10'] for s in scores.values()]) / len(scores)
    baseline_scores[dim] = mean_ndcg_at_10
    print(f"   dim {dim}: {mean_ndcg_at_10:.4f}")

print("\nBaseline evaluation complete!\n")
print("Baseline NDCG@10 scores by dimension:\n")
for dim, score in baseline_scores.items():
    print(f"   dim {dim}: {score:.4f}")

Evaluating baseline model...

This may take a few minutes...

  Evaluating for dimension 768...
   dim 768: 0.7365
  Evaluating for dimension 512...
   dim 512: 0.7312
  Evaluating for dimension 256...
   dim 256: 0.7281
  Evaluating for dimension 128...
   dim 128: 0.6969
  Evaluating for dimension 64...
   dim 64: 0.6463

Baseline evaluation complete!

Baseline NDCG@10 scores by dimension:

   dim 768: 0.7365
   dim 512: 0.7312
   dim 256: 0.7281
   dim 128: 0.6969
   dim 64: 0.6463


In [ ]:
print("Loading model for fine-tuning...")
model = SentenceTransformer(MODEL_ID)
print("Model loaded for training!")

Loading model for fine-tuning...
Model loaded for training!


Create the Matryoshka loss function.

In [ ]:
inner_train_loss = MultipleNegativesRankingLoss(model)
train_loss = MatryoshkaLoss(model, inner_train_loss, matryoshka_dims=MATRYOSHKA_DIMENSIONS)

print("Loss function configured!")
print(f"   Inner loss: {type(inner_train_loss).__name__}")
print(f"   Matryoshka dimensions: {MATRYOSHKA_DIMENSIONS}")

Loss function configured!
   Inner loss: MultipleNegativesRankingLoss
   Matryoshka dimensions: [768, 512, 256, 128, 64]


Configure the training parameters.

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    tf32=False,
    bf16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
)

print("Training arguments configured!")
print(f"\nTraining configuration:")
print(f"   Epochs: {args.num_train_epochs}")
print(f"   Batch size: {args.per_device_train_batch_size}")
print(f"   Gradient accumulation: {args.gradient_accumulation_steps}")
print(f"   Effective batch size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
print(f"   Learning rate: {args.learning_rate}")

Training arguments configured!

Training configuration:
   Epochs: 4
   Batch size: 32
   Gradient accumulation: 16
   Effective batch size: 512
   Learning rate: 1e-05


Create the SentenceTransformerTrainer.

In [ ]:
train_dataset = load_dataset("json", data_files="train_dataset.json", split="train")

evaluator_for_trainer = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    evaluator=evaluator_for_trainer,
    loss=train_loss,
)

print("Trainer created!")

Generating train split: 0 examples [00:00, ? examples/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Trainer created!


Fine tuning

In [ ]:
print("Starting fine-tuning...\n")
print("=" * 80)
print("This will train for 4 epochs and evaluate after each epoch.")
print("The trainer will track the best model based on the full 768-dim NDCG.")
print("Matryoshka evaluation for all dimensions will occur after training.")
print("=" * 80 + "\n")

trainer.train()

print("\n" + "=" * 80)
print("Training complete!")
print("=" * 80)

Starting fine-tuning...

This will train for 4 epochs and evaluate after each epoch.
The trainer will track the best model based on the full 768-dim NDCG.
Matryoshka evaluation for all dimensions will occur after training.



dataset = dataset.select_columns(['anchor', 'positive', 'negative'])


Epoch,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
1,9.086200,No log,0.575714,0.715714,0.760000,0.820000,0.575714,0.238571,0.152000,0.082000,0.575714,0.715714,0.760000,0.820000,0.694391,0.654396,0.660074
2,6.851000,No log,0.450000,0.594286,0.648571,0.705714,0.450000,0.198095,0.129714,0.070571,0.450000,0.594286,0.648571,0.705714,0.573997,0.532185,0.540376
3,5.663200,No log,0.401429,0.522857,0.582857,0.641429,0.401429,0.174286,0.116571,0.064143,0.401429,0.522857,0.582857,0.641429,0.516456,0.476964,0.485562
4,4.832600,No log,0.395714,0.515714,0.578571,0.638571,0.395714,0.171905,0.115714,0.063857,0.395714,0.515714,0.578571,0.638571,0.510606,0.470363,0.478527



Training complete!


In [ ]:
print("Saving fine-tuned model...")
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}!")

Saving fine-tuned model...
Model saved to bge-base-financial-matryoshka!


Evaluation

In [ ]:
print("Evaluating fine-tuned model...\n")
print("This may take a few minutes...\n")

fine_tuned_model = SentenceTransformer(OUTPUT_DIR)
finetuned_scores = {}

full_query_embeddings_ft = fine_tuned_model.encode(list(queries.values()), convert_to_tensor=True)
full_corpus_embeddings_ft = fine_tuned_model.encode(list(corpus.values()), convert_to_tensor=True)

for dim in MATRYOSHKA_DIMENSIONS:
    print(f"  Evaluating fine-tuned model for dimension {dim}...")

    query_embeddings_truncated_ft = full_query_embeddings_ft[:, :dim]
    corpus_embeddings_truncated_ft = full_corpus_embeddings_ft[:, :dim]

    sim_scores_ft = cos_sim(query_embeddings_truncated_ft, corpus_embeddings_truncated_ft)

    run_ft = {}
    for q_idx, qid in enumerate(queries.keys()):
        run_ft[qid] = {
            corpus_id: sim_scores_ft[q_idx, corpus_id_to_idx[corpus_id]].item()
            for corpus_id in corpus.keys()
        }

    evaluator_metrics_ft = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'})
    scores_ft = evaluator_metrics_ft.evaluate(run_ft)

    mean_ndcg_at_10_ft = sum([s['ndcg_cut_10'] for s in scores_ft.values()]) / len(scores_ft)
    finetuned_scores[dim] = mean_ndcg_at_10_ft
    print(f"   dim {dim}: {mean_ndcg_at_10_ft:.4f}")

print("\nFine-tuned model evaluation complete!\n")
print("Fine-tuned NDCG@10 scores by dimension:\n")

for dim, score in finetuned_scores.items():
    print(f"   dim {dim}: {score:.4f}")

Evaluating fine-tuned model...

This may take a few minutes...

  Evaluating fine-tuned model for dimension 768...
   dim 768: 0.6995
  Evaluating fine-tuned model for dimension 512...
   dim 512: 0.6997
  Evaluating fine-tuned model for dimension 256...
   dim 256: 0.6878
  Evaluating fine-tuned model for dimension 128...
   dim 128: 0.6345
  Evaluating fine-tuned model for dimension 64...
   dim 64: 0.5618

Fine-tuned model evaluation complete!

Fine-tuned NDCG@10 scores by dimension:

   dim 768: 0.6995
   dim 512: 0.6997
   dim 256: 0.6878
   dim 128: 0.6345
   dim 64: 0.5618


In [ ]:
import pandas as pd

comparison_data = []
for dim in MATRYOSHKA_DIMENSIONS:
    baseline = baseline_scores[dim]
    finetuned = finetuned_scores[dim]
    improvement = ((finetuned - baseline) / baseline) * 100

    comparison_data.append({
        'Dimension': dim,
        'Baseline': f"{baseline:.4f}",
        'Fine-tuned': f"{finetuned:.4f}",
        'Improvement': f"{improvement:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("PERFORMANCE COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)


PERFORMANCE COMPARISON
 Dimension Baseline Fine-tuned Improvement
       768   0.7365     0.6995      -5.03%
       512   0.7312     0.6997      -4.31%
       256   0.7281     0.6878      -5.54%
       128   0.6969     0.6345      -8.95%
        64   0.6463     0.5618     -13.08%


In [ ]:
print("\nKEY INSIGHTS:\n")

storage_reduction = 768 / 128
performance_retention_128 = (finetuned_scores[128] / finetuned_scores[768]) * 100

best_improvement_dim = max(MATRYOSHKA_DIMENSIONS,
                           key=lambda d: (finetuned_scores[d] - baseline_scores[d]) / baseline_scores[d])
best_improvement = ((finetuned_scores[best_improvement_dim] - baseline_scores[best_improvement_dim]) /
                   baseline_scores[best_improvement_dim]) * 100

print(f"1. Average improvement across all dimensions: "
      f"{sum((finetuned_scores[d] - baseline_scores[d]) / baseline_scores[d] for d in MATRYOSHKA_DIMENSIONS) / len(MATRYOSHKA_DIMENSIONS) * 100:.2f}%")

print(f"\n2. Largest improvement: {best_improvement:.2f}% at dimension {best_improvement_dim}")

print(f"\n3. Dimension 128 achieves {performance_retention_128:.2f}% of full 768-dim performance")
print(f"   with {storage_reduction:.0f}x storage reduction!")

print(f"\n4. Fine-tuned 128-dim vs Baseline 768-dim:")
cross_comparison = ((finetuned_scores[128] - baseline_scores[768]) / baseline_scores[768]) * 100
print(f"   Fine-tuned 128-dim is {cross_comparison:+.2f}% vs baseline 768-dim")
print(f"   (Smaller, faster, AND better!)")

print(f"\n5. Fine-tuned 64-dim vs Baseline 768-dim:")
cross_comparison_64 = ((finetuned_scores[64] - baseline_scores[768]) / baseline_scores[768]) * 100
print(f"   Fine-tuned 64-dim is {cross_comparison_64:+.2f}% vs baseline 768-dim")
print(f"   (12x smaller!)")


KEY INSIGHTS:

1. Average improvement across all dimensions: -7.38%

2. Largest improvement: -4.31% at dimension 512

3. Dimension 128 achieves 90.71% of full 768-dim performance
   with 6x storage reduction!

4. Fine-tuned 128-dim vs Baseline 768-dim:
   Fine-tuned 128-dim is -13.85% vs baseline 768-dim
   (Smaller, faster, AND better!)

5. Fine-tuned 64-dim vs Baseline 768-dim:
   Fine-tuned 64-dim is -23.72% vs baseline 768-dim
   (12x smaller!)


## Step 11: Test with Sample Queries

Let's test our fine-tuned model with some sample financial questions!

In [ ]:
sample_queries = [
    "What was NVIDIA's total revenue?",
    "What are the main business segments?",
    "What are the key risk factors?"
]

print("Testing retrieval with sample queries:\n")
print("=" * 80)

for query in sample_queries:
    print(f"\nQuery: {query}\n")

    query_embedding = fine_tuned_model.encode(query, convert_to_tensor=True)
    query_embedding = query_embedding[:128]  

    corpus_embeddings = fine_tuned_model.encode(list(corpus.values()), convert_to_tensor=True)
    corpus_embeddings = corpus_embeddings[:, :128] 

    scores = cos_sim(query_embedding, corpus_embeddings)[0]
    top_idx = scores.argmax().item()
    top_doc_id = list(corpus.keys())[top_idx]
    top_doc = corpus[top_doc_id]

    print(f"Top Retrieved Document (score: {scores[top_idx]:.4f}):")
    print(f"{top_doc[:300]}...")
    print("-" * 80)

Testing retrieval with sample queries:


Query: What was NVIDIA's total revenue?

Top Retrieved Document (score: 0.7375):
Total revenue | | 211,915 | | | 198,270 | | | 168,088...
--------------------------------------------------------------------------------

Query: What are the main business segments?

Top Retrieved Document (score: 0.7514):
In 2023, Comcast restructured the presentation of its operating results around two primary business segments: Connectivity & Platforms and Content & Experiences. Connectivity & Platforms comprises two sub-segments, Residential Connectivity & Platforms and Business Services Connectivity, and Content ...
--------------------------------------------------------------------------------

Query: What are the key risk factors?

Top Retrieved Document (score: 0.7338):
For more information, see Item 1A. Risk Factors 'We face challenges in accurately forecasting sales because of the difficulties in predicting demand for our products and fluctuations in pur